### **Dim Customers**

In [0]:
%sql
USE CATALOG dbacademy;
USE SCHEMA labuser14586003_1778076074;

In [0]:
%sql
CREATE OR REPLACE TABLE DimCustomers
AS
WITH rem_dup AS
(
SELECT 
  DISTINCT(customer_id),
  customer_email,
  customer_name,
  Customer_Name_Upper
FROM silver_table
)
SELECT * ,
      row_number() OVER (ORDER BY customer_id) as DimCustomerKey
FROM rem_dup

In [0]:
SELECT *
FROM dimcustomers;

### **Dim Products**

In [0]:
%sql
CREATE OR REPLACE TABLE DimProducts
AS
WITH rem_dup AS
(
SELECT 
  DISTINCT(product_id),
  product_name,
  product_category
FROM 
  silver_table
)
SELECT * ,
      row_number() OVER (ORDER BY product_id) as DimProductKey
FROM rem_dup

In [0]:
%sql
select * from dimproducts

### **Dim Payments**

In [0]:
%sql
CREATE OR REPLACE TABLE DimPayments
WITH rem_dup AS 
(
SELECT 
  DISTINCT(payment_type)
FROM silver_table
) 
SELECT *, row_number() OVER (ORDER BY payment_type) as DimPaymentKey FROM rem_dup

In [0]:
%sql
SELECT * FROM DimPayments

### **Dim Region**

In [0]:
%sql
CREATE OR REPLACE TABLE DimRegions
WITH rem_dup AS 
(
SELECT 
  DISTINCT(country)
FROM silver_table
) 
SELECT *, row_number() OVER (ORDER BY country) as DimRegionKey FROM rem_dup

In [0]:
%sql
SELECT * FROM dimregions

### **Dim Sales**

In [0]:
%sql
CREATE OR REPLACE TABLE DimSales
AS
SELECT 
 row_number() OVER (ORDER BY order_id) as DimSaleKey,
 order_id,
 order_date,
 customer_id,
 customer_name,
 customer_email,
 product_id,
 product_name,
 product_category,
 payment_type,
 country,
 last_updated,
 Customer_Name_Upper,
 processDate
FROM
  silver_table

In [0]:
%sql
SELECT * FROM dimsales

### **FACT TABLE**

In [0]:
%sql
CREATE OR REPLACE TABLE FactSales
AS
SELECT 
  S.DimSaleKey,
  C.DimCustomerKey,
  P.DimProductKey,
  R.DimRegionKey,
  PY.DimPaymentKey,
  F.quantity,
  F.unit_price
FROM silver_table F
LEFT JOIN 
  dimcustomers C
  ON F.customer_id = C.customer_id
LEFT JOIN 
  dimproducts P
  ON F.product_id = P.product_id
LEFT JOIN 
  dimregions R
  ON F.country = R.country
LEFT JOIN 
  dimpayments PY
  ON F.payment_type = PY.payment_type
LEFT JOIN 
  dimsales S
  ON F.order_id = S.order_id

In [0]:
%sql
SELECT * FROM FactSales